# Official ADK agent task trial
Run only in a private, Internet-disabled Gemma 4 Developer Agent competition L4×4 session. This distinct experiment evaluates the exact validated `submission.zip` against two public software-engineering tasks using the organizer's `swegemma` and vLLM serving stack. It does not use LiteLLM in the separate kaggle-vllm inference pipeline. No patch or pass is claimed until the task results and test logs are inspected.


In [ ]:
from pathlib import Path
import hashlib, json, os, subprocess, sys
python312='/usr/bin/python3.12'
assert Path(python312).is_file()
assert subprocess.run([python312,'-c','import sys;assert sys.version_info[:2]==(3,12)'],check=False).returncode==0
base = Path('/kaggle/input/datasets')
wheelhouse = base/'metric/gemma-4-developer-agent-wheelhouse'
agent_input = base/'waqasm86/gemma4-agent-v031-private-input'
runner = agent_input/'run_official_eval.py'
assert wheelhouse.is_dir() and agent_input.joinpath('submission/agent.yaml').is_file()
pip_candidates=sorted(Path('/kaggle/input').rglob('pip-25.2-py3-none-any.whl'))
print('pip_candidates',pip_candidates)
assert len(pip_candidates)==1, 'exact offline pip wheel missing or ambiguous'
pip_wheel=pip_candidates[0]
def sha256(path):
    digest=hashlib.sha256()
    with path.open('rb') as stream:
        for block in iter(lambda: stream.read(4*1024*1024), b''):digest.update(block)
    return digest.hexdigest()
assert sha256(runner) == '0afb5227b911f53a6cb4305dbaeb3f3f692789314d2517a8e56ff17145dd7809'
assert len(list(wheelhouse.glob('*.whl'))) == 41
manifest_candidates=[p for p in Path('/kaggle/input').rglob('WHEEL_MANIFEST.json') if 'gemma4-adk-litellm-18314-private' in p.parts]
assert len(manifest_candidates)==1, 'private ADK closure manifest missing or ambiguous'
manifest_path=manifest_candidates[0]
assert sha256(manifest_path)=='dfa087f197d1abbfd7572e7cd706bbd3e71279521f28a7a009d9a8b93c7c6d1d'
closure=json.loads(manifest_path.read_text())
assert closure['wheel_count']==len(closure['wheels'])==54
closure_files=[]
for item in closure['wheels']:
    path=manifest_path.parent/item['filename']
    assert path.is_file() and path.stat().st_size==item['size'] and sha256(path)==item['sha256'],path
    closure_files.append(path)
links=Path('/tmp/gemma4-official-wheels')
links.mkdir()
for source in wheelhouse.glob('*.whl'):
    name=source.name.replace('cu128','+cu128') if 'cu128' in source.name and '+' not in source.name else source.name
    (links/name).symlink_to(source)
target=Path('/tmp/gemma4-official-pydeps')
closure_target=Path('/tmp/gemma4-adk-closure-pydeps')
assert not target.exists() and not closure_target.exists(), 'Use a fresh Kaggle session'
installer_env=dict(os.environ); installer_env['PYTHONPATH']=str(pip_wheel)
subprocess.run([python312,'-m','pip','install','--no-index','--no-deps','--no-cache-dir','--no-compile','--target',str(target),*[str(p) for p in sorted(links.glob('*.whl'))]],check=True,timeout=1200,env=installer_env)
subprocess.run([python312,'-m','pip','install','--no-index','--no-deps','--no-cache-dir','--no-compile','--target',str(closure_target),*[str(p) for p in closure_files]],check=True,timeout=600,env=installer_env)
env=dict(os.environ)
env['PYTHONPATH']=os.pathsep.join(filter(None,(str(target),str(closure_target),env.get('PYTHONPATH',''))))
env.update({'HF_HUB_OFFLINE':'1','TRANSFORMERS_OFFLINE':'1','HF_DATASETS_OFFLINE':'1','VLLM_NO_USAGE_STATS':'1','VLLM_WORKER_MULTIPROC_METHOD':'spawn','VLLM_ENGINE_READY_TIMEOUT_S':'1200','NCCL_DEBUG':'WARN','LITELLM_LOCAL_MODEL_COST_MAP':'True','LITELLM_TELEMETRY':'False','OTEL_SDK_DISABLED':'true','OTEL_PYTHON_DISABLED':'true','LOCAL_API_KEY':'EMPTY'})
print('official_harness_staged',len(list(links.glob('*.whl'))),'official wheels and',len(closure_files),'ADK closure wheels under /tmp')


In [ ]:
from pathlib import Path
import subprocess
log=Path('/kaggle/working/06_official_agent_task_trial.log')
with log.open('w') as stream:
    result=subprocess.run([python312,str(runner)],env=env,stdout=stream,stderr=subprocess.STDOUT,timeout=5400)
print('runner_exit',result.returncode)
print('log_tail','\n'.join(log.read_text(errors='replace').splitlines()[-50:]))
assert result.returncode==0, 'Official trial failed; preserve logs and partial task results'
